In [1]:
import os
import pandas as pd
import numpy as np

# weighting method 비교: FP-only vs IMG-only vs 고정 커스텀 가중치(FP=0.75/IMG=0.25,
# FP=0.25/IMG=0.75) vs UNIFORM(고정 0.5/0.5) vs CUE(softmax, 학습된 modality 가중치).
# Random은 이 버전에서 제외한다 (v2까지의 swarmplot/p-value 비교 대신, 고정 가중치들끼리의
# 단순 비교로 구성).
cue_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue/results'

datasets = ['DIS', 'INH']
targets = ['PI', 'Ssel', 'WS2', 'RS2']
metrics_of_interest = ['rmse', 'r2', 'pcc']

# 비교할 소스: (표시 이름, metrics.csv 루트 경로). CUE는 항상 마지막에 오도록 순서를 고정한다.
model_sources = {
    'FP': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'FP'),
    'IMG': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'IMG'),
    'UNIFORM': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'UNIFORM'),
    'CUE': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'CUE'),
}

model_order = ['FP', 'IMG', 'UNIFORM', 'CUE']

# rmse는 낮을수록, r2/pcc는 높을수록 좋은 성능
LOWER_IS_BETTER = {'rmse': True, 'r2': False, 'pcc': False}

In [2]:
def load_metric(model_dir_template, dataset, target):
    """cmfe의 metrics.csv(train.py::aggregate_metrics 출력)를 읽어
    model_name == 'mlp' 행의 rmse/r2/pcc(+ _std)를 뽑아온다."""
    metrics_path = os.path.join(model_dir_template.format(dataset=dataset, target=target), 'metrics.csv')
    if not os.path.exists(metrics_path):
        return None

    df_m = pd.read_csv(metrics_path)
    mlp_row = df_m[df_m['model_name'] == 'mlp']
    if mlp_row.empty:
        return None

    values = {}
    for col in mlp_row.columns:
        if col in ['model_name', 'Unnamed: 0']:
            continue
        norm_col = col.lower()
        if norm_col in metrics_of_interest or norm_col.replace('_std', '') in metrics_of_interest:
            values[norm_col] = mlp_row[col].values[0]
    return values if values else None


raw_rows = []
for d_name in datasets:
    for target in targets:
        for model_name, dir_template in model_sources.items():
            vals = load_metric(dir_template, d_name, target)
            if vals is not None:
                raw_rows.append({'Dataset': d_name, 'Target': target, 'Model': model_name, **vals})

df_total = pd.DataFrame(raw_rows)
print(f"✅ 데이터 로드 완료: {len(df_total)}개의 항목이 수집되었습니다 (기대값: 2 x 4 x 6 = 48).")

✅ 데이터 로드 완료: 32개의 항목이 수집되었습니다 (기대값: 2 x 4 x 6 = 48).


In [3]:
METRIC_HEADERS = {'rmse': 'RMSE( ↓ )', 'r2': 'R2( ↑ )', 'pcc': 'PCC( ↑ )'}


def build_metric_table(df_all, dataset, target, metrics=metrics_of_interest, model_order=model_order):
    """지정한 dataset/target에 대해 Model별 metric 값('mean ± std')을 담은 표를 만든다 (순위 없음).

    Returns:
        Optional[pd.DataFrame]: 컬럼이 Model | RMSE( ↓ ) | R2( ↑ ) | PCC( ↑ ) 순서인 표.
            각 값은 '{mean:.3f} ± {std:.3f}' 문자열이다.
            해당 dataset/target에 데이터가 하나도 없으면 None.
    """
    df_sub = df_all[(df_all['Dataset'] == dataset) & (df_all['Target'] == target)]
    df_sub = df_sub.set_index('Model').reindex(model_order)
    df_sub = df_sub.dropna(subset=[m for m in metrics if m in df_sub.columns], how='all')
    if df_sub.empty:
        return None

    table = pd.DataFrame(index=df_sub.index)
    for m in metrics:
        if m not in df_sub.columns:
            continue
        header = METRIC_HEADERS.get(m, m.upper())
        if f'{m}_std' in df_sub.columns:
            table[header] = [f"{v:.3f} ± {s:.3f}" for v, s in zip(df_sub[m], df_sub[f'{m}_std'])]
        else:
            table[header] = df_sub[m].round(3)

    return table.reset_index()  # 'Model'을 첫 컬럼으로 복원


table_save_dir = os.path.join(cue_base, 'table', 'weight_comparison_v3')
os.makedirs(table_save_dir, exist_ok=True)

# DIS -> dis(dissociation), INH -> inh(inhibition)
dataset_filenames = {'DIS': 'dis.csv', 'INH': 'inh.csv'}

metric_tables = {}
for d_name in datasets:
    dataset_tables = []
    for target in targets:
        table = build_metric_table(df_total, d_name, target)
        if table is None:
            continue
        metric_tables[(d_name, target)] = table

        print(f"\n=== [{d_name} / {target}] ===")
        print(table.to_string(index=False))

        # 맨 처음 컬럼에 Target을 추가한 뒤(targets 순서 = PI, Ssel, WS2, RS2) 이어 붙인다.
        table_with_target = table.copy()
        table_with_target.insert(0, 'Target', target)
        dataset_tables.append(table_with_target)

    if not dataset_tables:
        continue

    dataset_table = pd.concat(dataset_tables, ignore_index=True)
    save_path = os.path.join(table_save_dir, dataset_filenames[d_name])
    dataset_table.to_csv(save_path, index=False, encoding='utf-8-sig', sep='\t')
    print(f"\n✅ 저장 완료: {save_path}")


=== [Kd / PI] ===
  Model     RMSE( ↓ )       R2( ↑ )      PCC( ↑ )
     FP 0.245 ± 0.012 0.098 ± 0.059 0.357 ± 0.085
    IMG 0.257 ± 0.007 0.008 ± 0.043 0.194 ± 0.084
UNIFORM 0.242 ± 0.010 0.121 ± 0.059 0.361 ± 0.097
    CUE 0.242 ± 0.010 0.125 ± 0.055 0.366 ± 0.092

=== [Kd / Ssel] ===
  Model     RMSE( ↓ )        R2( ↑ )      PCC( ↑ )
     FP 0.887 ± 0.042  0.081 ± 0.073 0.326 ± 0.072
    IMG 0.957 ± 0.105 -0.072 ± 0.188 0.189 ± 0.095
UNIFORM 0.901 ± 0.074  0.051 ± 0.118 0.303 ± 0.110
    CUE 0.895 ± 0.068  0.065 ± 0.107 0.314 ± 0.104

=== [Kd / WS(alpha=2)] ===
  Model     RMSE( ↓ )       R2( ↑ )      PCC( ↑ )
     FP 0.285 ± 0.018 0.203 ± 0.052 0.470 ± 0.044
    IMG 0.300 ± 0.009 0.114 ± 0.098 0.380 ± 0.096
UNIFORM 0.282 ± 0.012 0.215 ± 0.069 0.488 ± 0.057
    CUE 0.282 ± 0.014 0.221 ± 0.061 0.490 ± 0.052

=== [Kd / RS(k=2)] ===
  Model     RMSE( ↓ )        R2( ↑ )      PCC( ↑ )
     FP 1.737 ± 0.195 -0.053 ± 0.122 0.299 ± 0.039
    IMG 1.806 ± 0.280 -0.131 ± 0.167 0.113 ± 0.046
